In [1]:
import sys

In [2]:
# sys.path.append("/home/gputnam/Diffusion-Anomaly-Detection/diffusion-anomaly")
sys.path.append("anomaly-detection/diffusion-anomaly")

In [3]:
from guided_diffusion.script_util import (
    model_and_diffusion_defaults,
    diffusion_defaults,
    create_model_and_diffusion,
    args_to_dict,
    add_dict_to_argparser,
    create_gaussian_diffusion
)

from guided_diffusion.resample import UniformSampler
from guided_diffusion import dist_util

from guided_diffusion.fp16_util import *

from guided_diffusion.respace import space_timesteps

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch as th

import h5py

import os

import pickle

In [4]:
plt.rcParams.update({'font.size': 14})

In [3]:
import os
print(os.environ["LD_PRELOAD"])
def pnfs2xrootd(fname):
    return fname.replace("/pnfs", "root://fndcadoor.fnal.gov:1094/pnfs/fnal.gov/usr")
    

In [6]:
def visualize(img):
    return np.clip(img, -1, 1)

def events_from_file(intresting_fname):
    with h5py.File(pnfs2xrootd(intresting_fname), "r") as f:
        # scale charge by 500
        arrs = [f[ev]["raw"][:] for ev in f.keys()]
    allarrs = []
    for arr in arrs:
        nrows, ncols = (512, 512)
        plane_boundaries = [0, 1984, 3968, 5638]
        for planeno, (wlo, whi) in enumerate(zip(plane_boundaries[:-1], plane_boundaries[1:])):
            planearr = arr[wlo:whi, :]
            h, w = planearr.shape

            planearr_cropped = planearr[:(h // nrows) * nrows, :(w // nrows) * nrows]
            H_crop, W_crop = planearr_cropped.shape
            n_patches_h = H_crop // nrows
            n_patches_w = W_crop // ncols
            ll = planearr_cropped.reshape(n_patches_h, nrows, n_patches_w, ncols).swapaxes(1, 2).reshape(-1, nrows, ncols)

            patch_layout = {
                "patch_size": (nrows, ncols),
                "grid_shape": (n_patches_h, n_patches_w),
                "cropped_shape": (H_crop, W_crop),
                "full_plane_shape": (h, w),
            }

            cscale = [200., 100., 200.][planeno]
            allarrs.append(ll/cscale)
            
    return visualize(np.expand_dims(np.concatenate(allarrs), axis=1)).astype(np.float32), patch_layout

In [7]:
def get_event_displays(rows, df):
    #filters the intresting events list by row number and makes 3 arrays of the 
    #reasons, frame numbers and filenames for each intresting event
    filtered = df.iloc[rows]
    frame_numbers = filtered['frame number'].to_numpy()
    reasons = filtered['reason for flag'].to_numpy()
    file_names = filtered['filename'].to_numpy()

    interesting_imgs = []
    for i in range(len(rows)):
        arrs, patch_layout = events_from_file(file_names[i])
        interesting_imgs.append(arrs[frame_numbers[i]].T)

    return interesting_imgs, reasons, patch_layout
        

In [14]:
!. ~/get_token.sh

In [15]:
interesting_df = pd.read_csv("/exp/sbnd/app/users/munjung/anomaly-detection/interesting_list.csv")
print(interesting_df[['frame number','reason for flag']])

In [16]:
#row numbers ofevents to process
#look above and chose which events from list are intresting 
row_numbers = [0,4,6,8]
interesting_imgs, reasons, patch_layout = get_event_displays(row_numbers, interesting_df)

In [ ]:
for i in range(len(row_numbers)):
    plt.figure(i,figsize=(5,5))
    plt.title(reasons[i])
    plt.imshow(np.squeeze(interesting_imgs[i]), vmin=-0.5, vmax=0.5)

In [ ]:
imgs = np.expand_dims(np.stack(interesting_imgs), axis=1)
#imgs = np.array(interesting_imgs)

print(imgs.shape)
imgs = th.tensor(imgs, device=dist_util.dev())

In [ ]:
th.set_grad_enabled(False)

In [ ]:
args = model_and_diffusion_defaults()
diffusion_args = diffusion_defaults()

In [ ]:
# MODEL
args["image_size"] = 512
args["num_channels"] = 32
args["class_cond"] = False
args["num_res_blocks"] = 2
args["num_heads"] = 8
args["learn_sigma"] = True
args["use_scale_shift_norm"] = False
args["attention_resolutions"] = "16,32"
args["channel_mult"] = "1,2,4,8,8,8"

# DIFFUSION
diffusion_args["diffusion_steps"] = 1000
diffusion_args["noise_schedule"] = "linear"
diffusion_args["rescale_learned_sigmas"] = False
diffusion_args["rescale_timesteps"] = False

diffusion_args.pop("diffusion_steps")
diffusion_args.pop("timestep_respacing")

# TODO: change?
diffusion_args["learn_sigma"] = True

args = args | diffusion_args

In [ ]:
model, diffusion = create_model_and_diffusion(**args)

In [ ]:
MODEL = "/exp/sbnd/data/users/gputnam/training-SBND/iterE/results/brats2update111000.pt"

In [ ]:
model.load_state_dict(
    dist_util.load_state_dict(MODEL, map_location="cpu")
)

model.to(dist_util.dev())
_ = model.eval()

In [ ]:
T = 100

In [ ]:
imgs = imgs.squeeze(-1)
#if i get dimention warning squeeze dimentions with above command
ddim_noisef = diffusion.ddim_sample_loop_progressive(model, imgs.shape, time=T, noise=imgs, 
                                             reverse=True, progress=True)

ddim_noise = list(ddim_noisef)

In [ ]:
ddim_noised = ddim_noise[-1]["sample"]

In [ ]:
rand_noised = diffusion.q_sample(imgs, th.tensor(T, device=dist_util.dev()))

In [ ]:
ddim_2_ddim = diffusion.ddim_sample_loop_progressive(model, 
                imgs.shape, time=T, noise=ddim_noised, progress=True)

rand_2_ddim = diffusion.ddim_sample_loop_progressive(model, 
                imgs.shape, time=T, noise=rand_noised, progress=True)

ddim_2_ddpm = diffusion.p_sample_loop_progressive(model, 
                imgs.shape, time=T, noise=ddim_noised, progress=True)

rand_2_ddpm = diffusion.p_sample_loop_progressive(model, 
                imgs.shape, time=T, noise=rand_noised, progress=True)



In [ ]:
ddim_2_ddim_reco = list(ddim_2_ddim)[-1]["sample"]
rand_2_ddim_reco = list(rand_2_ddim)[-1]["sample"]
# ddim_2_ddpm_reco = list(ddim_2_ddpm)[-1]["sample"]
rand_2_ddpm_reco = list(rand_2_ddpm)[-1]["sample"]

In [ ]:
recos = [
    ddim_2_ddim_reco,
    rand_2_ddim_reco,
    # ddim_2_ddpm_reco,
    rand_2_ddpm_reco
]

In [ ]:
titles = reasons
#or make our own titles list that is cleaner than reasons and is a better title

reco_names = [
    "DDIM to DDIM",
    "Random to DDIM",
    # "DDIM to DDPM",
    "Random to DDPM",
]

In [ ]:
for ifig, title in enumerate(titles):
    plt.figure(ifig)
    fig, axs = plt.subplots(2, 4, figsize=(9.6,5))

    axs[0][0].set_title("Original Image")
    axs[0][0].imshow(np.squeeze(imgs[ifig].cpu().numpy()), vmin=-0.5, vmax=0.5)
    axs[1][0].axis('off')

    axs[1][0].text(-0.3, 0.5, title.replace(" ", "\n"), horizontalalignment="left", 
                   verticalalignment="center", fontsize=18, fontweight="bold")
    for ireco, rname in enumerate(reco_names):
        axs[0][ireco+1].imshow(np.squeeze(recos[ireco][ifig].cpu().numpy()), vmin=-0.5, vmax=0.5)
        axs[1][ireco+1].imshow(np.squeeze((imgs[ifig] - recos[ireco][ifig]).cpu().numpy()), vmin=-0.1, vmax=0.1, cmap="bwr")
        
        axs[0][ireco+1].set_xticks([])
        axs[0][ireco+1].set_yticks([])
        
        if ireco > 0:
            axs[1][ireco+1].set_yticks([])
            
        axs[0][ireco+1].set_title(rname, fontsize=12)

    
    fig.subplots_adjust(wspace=0.02, hspace=0.2) # Make room on the right
    fig.suptitle("Reconstructed Images", x=0.62)
    axs[1][2].set_title("Sailiency Maps")
    
    axs[0][0].set_xlabel("Wire Number")
    axs[0][0].set_ylabel("Time Tick")



## Stick Patches

In [ ]:
def stitch_patches(patches, grid_shape, patch_size=None):

    nh, nw = grid_shape

    if patch_size is None:
        ph, pw = int(patches.shape[-2]), int(patches.shape[-1])
    else:
        ph, pw = patch_size

    if isinstance(patches, th.Tensor):
        if patches.dim() == 4:
            n, c, _, _ = patches.shape
            x = patches.view(nh, nw, c, ph, pw).permute(2, 0, 3, 1, 4).reshape(c, nh * ph, nw * pw)
            return x
        if patches.dim() == 3:
            x = patches.view(nh, nw, ph, pw).permute(0, 2, 1, 3).reshape(nh * ph, nw * pw)
            return x

    if patches.ndim == 4:
        n, c, _, _ = patches.shape
        return patches.reshape(nh, nw, c, ph, pw).transpose(0, 2, 1, 3, 4).reshape(c, nh * ph, nw * pw)

    if patches.ndim == 3:
        return patches.reshape(nh, nw, ph, pw).swapaxes(1, 2).reshape(nh * ph, nw * pw)

In [ ]:
original = stitch_patches(imgs.cpu().squeeze(1).numpy(), patch_layout["grid_shape"])
plt.imshow(original, vmin=-0.5, vmax=0.5, aspect="auto")
plt.show();

full_reco = stitch_patches(ddim_2_ddim_reco.cpu().squeeze(1).numpy(), patch_layout["grid_shape"])
plt.imshow(full_reco, vmin=-0.5, vmax=0.5, aspect="auto")
plt.show();

sal_map = full_reco - original
plt.imshow(sal_map, vmin=-0.5, vmax=0.5, aspect="auto")
plt.show();